# 05 — Measured image and pore statistics

Maintained workflow. Default examples are synthetic smoke tests—not thesis results. Install `pip install -e '.[learning,notebook]'` first. Only maintained notebook versions are included in this repository.


In [ ]:
from pathlib import Path
import numpy as np

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'configs' / 'demo.json').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError('Start Jupyter inside the repository after installing the package.')


In [ ]:
from porous_media.preparation import synthetic_rock
from porous_media.volumes import IntensityScaler
from porous_media.metrics import reconstruction_metrics, pore_mask, porosity, axial_two_point

DEMO = True
if DEMO:
    raw, labels = synthetic_rock((16, 16, 16))
    reference = IntensityScaler.fit(raw).transform(raw)
    prediction = np.clip(reference + np.random.default_rng(42).normal(0, 0.02, reference.shape), -1, 1)
else:
    reference = np.load(REPO_ROOT / 'outputs' / 'reference.npy', allow_pickle=False)
    prediction = np.load(REPO_ROOT / 'outputs' / 'prediction.npy', allow_pickle=False)

report = reconstruction_metrics(reference, prediction, data_range=2.0)
threshold = 0.0  # Explicit shared policy for this synthetic example, not a rock pore threshold.
reference_pores = pore_mask(reference, threshold=threshold)
predicted_pores = pore_mask(prediction, threshold=threshold)
print('Synthetic:', DEMO)
print(report)
print('Porosity:', porosity(reference_pores), porosity(predicted_pores))
lag, probability = axial_two_point(reference_pores, axis=0, max_lag=8)
print('Lag:', lag, 'Two-point probability:', probability)


Use a physically justified pore-label or threshold policy for real images and record it. Fixed data range must match the intensity convention (2 for [-1,1]). Axial two-point probability is not a radial correlation estimate. SNOW2 extraction and OpenPNM flow require the optional flow environment, provider voxel size in meters, validated topology and explicit geometry/phase models; see `docs/flow-validation.md`. Do not substitute typed published numbers for computed measurements.
